In [ ]:

# Load model directly
from transformers import AutoTokenizer, AutoModelForCausalLM
from transformers import BitsAndBytesConfig
import torch

# quant_config = BitsAndBytesConfig(
#     load_in_4bit=True,
#     bnb_4bit_quant_type="nf4",
#     bnb_4bit_compute_dtype=torch.float16,
#     bnb_4bit_use_double_quant=False
# )

quant_config = BitsAndBytesConfig(
    load_in_8bit=True,
    llm_int8_threshold=6.0
)

# tokenizer = AutoTokenizer.from_pretrained("EleutherAI/llemma_7b")
# model = AutoModelForCausalLM.from_pretrained("EleutherAI/llemma_7b", quantization_config=quant_config, device_map={"": 0})


# tokenizer = AutoTokenizer.from_pretrained("meta-llama/Meta-Llama-3-8B-Instruct")
# model = AutoModelForCausalLM.from_pretrained("meta-llama/Meta-Llama-3-8B-Instruct", quantization_config=quant_config,
#                                              device_map={"": 0})


tokenizer = AutoTokenizer.from_pretrained('deepseek-ai/deepseek-math-7b-rl')
model = AutoModelForCausalLM.from_pretrained('deepseek-ai/deepseek-math-7b-rl', quantization_config=quant_config, device_map='auto')


# filename = 'deepseek-math-7b-rl.Q8_0.gguf'
# tokenizer = AutoTokenizer.from_pretrained('QuantFactory/deepseek-math-7b-rl-GGUF')#, gguf_file=filename)
# model = AutoModelForCausalLM.from_pretrained('QuantFactory/deepseek-math-7b-rl-GGUF', gguf_file=filename, device_map={"": 0})




In [ ]:
tokenizer.pad_token_id = tokenizer.eos_token_id

In [ ]:
model.generation_config.pad_token_id = tokenizer.pad_token_id

In [ ]:
# state = '\"\"\"Given the Lean 4 tactic state, suggest a next tactic. Do NOT show working"\n\n\
state = 'α : Type u_1\n\
r : α → α → Prop\n\
inst1 : DecidableEq α\n\
inst : IsIrrefl α r\n\
⊢ CutExpand r ≤ InvImage (Finsupp.Lex (cr Π fun x x_1 => x̸ = x_1)\n\
fun x x_1 => x < x_1) ↑toFinsupp\n\
---\n\n\
Next tactic:'

tokenized_state = tokenizer(
    state,
    padding="longest",
    max_length=2300,
    truncation=True,
    return_tensors="pt",
)

state_ids = tokenized_state.input_ids.cuda()
state_mask = tokenized_state.attention_mask.cuda()


In [ ]:
with torch.no_grad():
    num_samples = 10
    output = model.generate(
        input_ids=state_ids,
        attention_mask=state_mask,
        max_new_tokens=50,
        top_k =50,
        top_p = 0.95,
        num_return_sequences=num_samples,
        do_sample=True,
        # length_penalty=1.0
    )

In [ ]:
tokenizer.batch_decode(output)